# Burning Cost in 30 Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/burning-cost-examples/blob/main/notebooks/burning_cost_30_minutes.ipynb)

This notebook runs a complete mini-workflow using three libraries from the Burning Cost ecosystem:

| Library | What it does |
|---|---|
| `insurance-causal` | Double Machine Learning to estimate causal effects on observational data |
| `insurance-conformal` | Distribution-free prediction intervals with coverage guarantees |
| `insurance-monitoring` | PSI/CSI drift detection, A/E monitoring, Gini discrimination tracking |

**No data required.** Everything runs on a 5,000-row synthetic UK motor dataset generated below.

**Runtime: ~5 minutes on Colab free tier.**

---

### The scenario

You are a pricing actuary at a UK motor insurer. You have:

- A fitted CatBoost frequency model (Poisson)
- A question: *does opting into telematics actually reduce claim frequency, or do safer drivers self-select into telematics?*
- A need for uncertainty quantification around frequency predictions
- A monitoring pack due at quarter-end

All three problems are addressed below.

## 1. Install libraries

This takes 2-3 minutes on Colab. Go make a cup of tea.

In [ ]:
# Install the three flagship libraries.
# insurance-causal pulls in doubleml and econml; insurance-conformal and
# insurance-monitoring are lightweight — numpy/scipy/polars only.
!pip install -q insurance-causal insurance-conformal insurance-monitoring catboost
print("Done.")

## 2. Generate synthetic UK motor data

5,000 policies with realistic UK motor features. The data generating process has
a deliberate confounding problem: drivers who opt into telematics also tend to
be younger and lower-mileage. Naive regression will underestimate the true causal
effect of telematics because those risk factors are correlated with both the
treatment (opt-in) and the outcome (claim frequency).

We split the data into a **training period** (first 3,500 rows) and a
**monitoring period** (last 1,500 rows, with simulated distribution shift).
This mirrors the standard insurance setup: model built on historical data,
monitored on current in-force.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)
N = 5_000
N_TRAIN = 3_500

# --- Rating factors ---
driver_age = rng.integers(18, 75, size=N).astype(float)
vehicle_group = rng.choice([1, 2, 3, 4, 5, 6, 7, 8], size=N,
                            p=[0.08, 0.12, 0.18, 0.20, 0.18, 0.12, 0.07, 0.05])
ncd_years = rng.choice([0, 1, 2, 3, 4, 5], size=N,
                        p=[0.10, 0.10, 0.15, 0.20, 0.20, 0.25])
postcode_area = rng.choice(
    ["London", "SE England", "Midlands", "North", "Scotland", "Wales"],
    size=N,
    p=[0.18, 0.16, 0.20, 0.22, 0.14, 0.10],
)
annual_mileage = rng.lognormal(mean=9.2, sigma=0.5, size=N).clip(1_000, 50_000)
exposure = rng.uniform(0.5, 1.0, size=N)  # car-years

# --- True log-rate from risk factors ---
postcode_effect = {
    "London": 0.25, "SE England": 0.10, "Midlands": 0.05,
    "North": 0.00, "Scotland": -0.10, "Wales": -0.05,
}
log_rate = (
    -2.5
    + 0.012 * np.maximum(0, 25 - driver_age)  # young driver loading
    - 0.008 * np.maximum(0, driver_age - 55)  # older driver loading
    + 0.08 * vehicle_group
    - 0.07 * ncd_years
    + np.array([postcode_effect[p] for p in postcode_area])
    + 0.00015 * annual_mileage
)

# --- Telematics opt-in (confounded by risk factors) ---
# Younger, lower-mileage drivers are more likely to opt in.
# This creates selection bias: naive estimate overstates the benefit of telematics.
logit_telematics = (
    1.2
    - 0.04 * (driver_age - 35)  # younger more likely to opt in
    - 0.00008 * annual_mileage  # low-mileage more likely
    + 0.05 * (5 - ncd_years)    # newer drivers more price-sensitive, seek discount
)
p_telematics = 1 / (1 + np.exp(-logit_telematics))
telematics_optin = rng.binomial(1, p_telematics)

# True causal effect of telematics: -0.15 on log frequency (about -14% fewer claims)
TRUE_CAUSAL_EFFECT = -0.15
log_rate_with_treatment = log_rate + TRUE_CAUSAL_EFFECT * telematics_optin

# --- Observed claim counts ---
mu = np.exp(log_rate_with_treatment) * exposure
claim_count = rng.poisson(mu)

# Claim amounts: only for policies with at least one claim
claim_amount = np.where(
    claim_count > 0,
    rng.lognormal(7.5, 0.8, size=N) * claim_count,
    0.0,
)

df = pd.DataFrame({
    "driver_age": driver_age,
    "vehicle_group": vehicle_group.astype(float),
    "ncd_years": ncd_years.astype(float),
    "postcode_area": postcode_area,
    "annual_mileage": annual_mileage,
    "exposure": exposure,
    "telematics_optin": telematics_optin.astype(float),
    "claim_count": claim_count,
    "claim_amount": claim_amount,
})

# --- Split into training and monitoring periods ---
# Monitoring period has a mild distribution shift: slightly older book, higher mileage
df_train = df.iloc[:N_TRAIN].copy()

df_monitor = df.iloc[N_TRAIN:].copy()
df_monitor["driver_age"] = (df_monitor["driver_age"] + rng.normal(2, 1, size=len(df_monitor))).clip(18, 80)
df_monitor["annual_mileage"] = (df_monitor["annual_mileage"] * rng.lognormal(0.05, 0.05, size=len(df_monitor))).clip(1_000, 50_000)

print(f"Training period: {len(df_train):,} policies")
print(f"Monitoring period: {len(df_monitor):,} policies")
print(f"\nOverall claim frequency: {claim_count.sum() / exposure.sum():.4f} claims per car-year")
print(f"Telematics opt-in rate: {telematics_optin.mean():.1%}")
print(f"\nFirst 5 rows:")
df.head()

## 3. Fit a CatBoost frequency model

Standard Poisson GBM on training data. We hold out 20% of the training period
for conformal calibration. The model learns log-rate from all features including
telematics — but this rate estimate is confounded. The causal step below isolates
the pure treatment effect.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

FEATURES = ["driver_age", "vehicle_group", "ncd_years", "annual_mileage"]
# Excluding postcode_area (categorical, would need encoding) and telematics
# (confounded treatment — isolated below via DML)

X_train_full = df_train[FEATURES].values
y_train_full = df_train["claim_count"].values
w_train_full = df_train["exposure"].values

# Split training data: model fit vs conformal calibration
idx = np.arange(len(X_train_full))
idx_fit, idx_cal = train_test_split(idx, test_size=0.20, random_state=42)

X_fit = X_train_full[idx_fit]
y_fit = y_train_full[idx_fit]
w_fit = w_train_full[idx_fit]

X_cal = X_train_full[idx_cal]
y_cal = y_train_full[idx_cal]
w_cal = w_train_full[idx_cal]

# CatBoost Poisson GBM
# offset = log(exposure) is the standard Poisson GLM/GBM setup for frequency models
model = CatBoostRegressor(
    loss_function="Poisson",
    iterations=300,
    depth=5,
    learning_rate=0.05,
    random_seed=42,
    verbose=0,
)
model.fit(
    X_fit, y_fit,
    sample_weight=w_fit,
)

# In-sample A/E check on calibration set
preds_cal = model.predict(X_cal)
# Scale predictions by exposure to get expected claim counts
expected_cal = preds_cal * w_cal
ae_train = y_cal.sum() / expected_cal.sum()

print(f"CatBoost Poisson GBM fitted on {len(X_fit):,} policies")
print(f"Calibration set A/E ratio: {ae_train:.3f}  (1.00 = perfect balance)")
print(f"\nModel features: {FEATURES}")

## 4. Causal inference: does telematics reduce claims?

**The problem with naive regression:** telematics opt-in is correlated with driver
age and mileage. Younger drivers opt in more. Younger drivers also have higher claim
frequencies for unrelated reasons. A naive comparison of opt-in vs non-opt-in
claim rates mixes the *causal effect of telematics* with the *selection effect*
(safer drivers choosing telematics).

**Double Machine Learning** (Chernozhukov et al. 2018) fixes this. It:
1. Fits ML models for $E[Y|X]$ and $E[D|X]$ separately (the nuisance models)
2. Takes residuals: $\tilde{Y} = Y - \hat{E}[Y|X]$, $\tilde{D} = D - \hat{E}[D|X]$
3. Regresses $\tilde{Y}$ on $\tilde{D}$ — this regression is now free of confounding

The result is root-n-consistent and asymptotically normal: a valid causal estimate
with confidence intervals.

The true causal effect in our DGP is **-0.15** (telematics reduces log-frequency
by 0.15, roughly 14% fewer claims). The naive estimate will be biased toward a
larger reduction because low-risk drivers self-select.

In [ ]:
from insurance_causal import CausalPricingModel
from insurance_causal.treatments import BinaryTreatment

CONFOUNDERS = ["driver_age", "vehicle_group", "ncd_years", "annual_mileage"]

causal_model = CausalPricingModel(
    outcome="claim_count",
    outcome_type="poisson",          # divides claim count by exposure before DML
    treatment=BinaryTreatment(
        column="telematics_optin",
        positive_label="opted in",
        negative_label="not opted in",
    ),
    confounders=CONFOUNDERS,
    exposure_col="exposure",
    cv_folds=3,                      # 3-fold cross-fitting for speed
    random_state=42,
)

print("Fitting DML model (cross-fitting nuisance models)...")
causal_model.fit(df_train)
print("Done.")

In [ ]:
ate = causal_model.average_treatment_effect()
print(ate)
print()

# Naive estimate: simple difference in mean claim frequencies
freq_train = df_train["claim_count"] / df_train["exposure"]
naive_diff = (
    freq_train[df_train["telematics_optin"] == 1].mean()
    - freq_train[df_train["telematics_optin"] == 0].mean()
)

print(f"Comparison:")
print(f"  True causal effect (DGP):  {TRUE_CAUSAL_EFFECT:.4f}")
print(f"  DML estimate:              {ate.estimate:.4f}  (95% CI: [{ate.ci_lower:.4f}, {ate.ci_upper:.4f}])")
print(f"  Naive difference in means: {naive_diff:.4f}")
print()
print(f"The naive estimate is {abs(naive_diff - TRUE_CAUSAL_EFFECT) / abs(TRUE_CAUSAL_EFFECT):.0%} off the true effect.")
print(f"DML recovers the true causal effect within its confidence interval: ", end="")
print(ate.ci_lower <= TRUE_CAUSAL_EFFECT <= ate.ci_upper)

### What does this mean for pricing?

The DML estimate tells you the **causal** effect of telematics on claim frequency,
purged of selection bias. If you were using the naive estimate to set telematics
discounts, you would be offering discounts that are too large — compensating drivers
not just for driving carefully, but also for being inherently lower risk.

The DML estimate is what you should use to set the telematics loading/discount
in your rating structure. The naive estimate is commercially wrong.

## 5. Conformal prediction intervals

Point predictions from the CatBoost model tell you the expected claim count.
But pricers need to know: *how uncertain is this prediction?*

Bootstrap intervals are expensive and require refitting. Parametric Poisson intervals
assume the model is correctly specified (it isn't). **Split conformal prediction**
makes no distributional assumptions — it uses the calibration set to compute
prediction intervals with a finite-sample marginal coverage guarantee:

$$P(Y_{\text{new}} \in [\hat{l}(X), \hat{u}(X)]) \geq 1 - \alpha$$

For Poisson/count data, the right non-conformity score is the **Pearson-weighted
residual**: $(y - \hat{y}) / \hat{y}^{p/2}$. This accounts for the heteroscedasticity
in count data — high-risk policies have higher variance — giving narrower intervals
for low-risk policies and appropriately wide intervals for high-risk ones.

We use `alpha=0.10` for 90% coverage intervals.

In [ ]:
from insurance_conformal import InsuranceConformalPredictor

cp = InsuranceConformalPredictor(
    model=model,
    nonconformity="pearson_weighted",  # correct score for Poisson/count data
    distribution="poisson",
    tweedie_power=1.0,                 # Poisson = Tweedie(p=1)
)

# Calibrate on the held-out calibration set
cp.calibrate(X_cal, y_cal)

print(f"Calibrated on {cp.n_calibration_:,} observations")
print(f"Calibration quantile (alpha=0.10): {cp._get_quantile(0.10):.4f}")

In [ ]:
# Generate 90% prediction intervals for the monitoring period
X_monitor = df_monitor[FEATURES].values
y_monitor = df_monitor["claim_count"].values

intervals = cp.predict_interval(X_monitor, alpha=0.10)

# Check coverage on monitoring period (out-of-distribution due to the shift)
y_np = y_monitor
covered = (
    (y_np >= intervals["lower"].to_numpy()) &
    (y_np <= intervals["upper"].to_numpy())
)

print(f"Prediction interval summary (monitoring period, alpha=0.10):")
print(f"  Target coverage:        90.0%")
print(f"  Achieved coverage:      {covered.mean():.1%}")
print(f"  Mean interval width:    {(intervals['upper'] - intervals['lower']).mean():.4f} claims")
print()
print("Sample intervals (first 8 policies):")
sample = intervals.head(8).with_columns(
    actual=y_monitor[:8]
)
print(sample)

In [ ]:
# Coverage by risk decile
# This is the key diagnostic. Marginal coverage (above) can be correct on average
# while being systematically wrong for high-risk policies. The decile check exposes that.

decile_coverage = cp.coverage_by_decile(X_monitor, y_monitor, alpha=0.10)
print("Coverage by predicted risk decile:")
print(decile_coverage)

## 6. Model drift monitoring

The monitoring period has a mild distribution shift: drivers are slightly older
and have higher mileage than the training period. This is realistic — books age,
mix shifts over time.

The `MonitoringReport` runs three checks in one call:

| Check | What it detects |
|---|---|
| **A/E ratio** | Calibration drift — model systematically over or underpredicts |
| **Gini coefficient** | Discrimination drift — model's rank-ordering has degraded |
| **CSI** | Feature drift — which rating factors have shifted |

Traffic lights use the arXiv 2510.04556 thresholds: green / amber / red.
The recommendation follows the three-stage decision tree: NO_ACTION, RECALIBRATE,
or REFIT.

In [ ]:
import polars as pl
from insurance_monitoring import MonitoringReport

# Predictions for both periods
X_train_arr = df_train[FEATURES].values
preds_train = model.predict(X_train_arr)
preds_monitor = model.predict(X_monitor)

# Feature DataFrames for CSI
numeric_features = ["driver_age", "vehicle_group", "ncd_years", "annual_mileage"]
feat_train = pl.from_pandas(df_train[numeric_features])
feat_monitor = pl.from_pandas(df_monitor[numeric_features])

report = MonitoringReport(
    reference_actual=df_train["claim_count"].values,
    reference_predicted=preds_train * df_train["exposure"].values,
    current_actual=df_monitor["claim_count"].values,
    current_predicted=preds_monitor * df_monitor["exposure"].values,
    exposure=df_monitor["exposure"].values,
    reference_exposure=df_train["exposure"].values,
    feature_df_reference=feat_train,
    feature_df_current=feat_monitor,
    features=numeric_features,
    murphy_distribution="poisson",   # Murphy decomposition: RECALIBRATE vs REFIT
    n_bootstrap=100,                 # reduced for Colab speed
)

print(f"Monitoring recommendation: {report.recommendation}")
print()

In [ ]:
# A/E ratio
ae = report.results_["ae_ratio"]
print(f"A/E ratio: {ae['value']:.3f}  [{ae['band'].upper()}]")
print(f"  95% CI: [{ae['lower_ci']:.3f}, {ae['upper_ci']:.3f}]")
print(f"  Actual claims: {ae['n_claims']:.1f}   Expected: {ae['n_expected']:.1f}")
print()

# Gini coefficient
gini = report.results_["gini"]
print(f"Gini (reference): {gini['reference']:.3f}")
print(f"Gini (current):   {gini['current']:.3f}  [{gini['band'].upper()}]")
print(f"  Change: {gini['change']:+.3f}   p-value: {gini['p_value']:.3f}")
print()

In [ ]:
# CSI per feature
import polars as pl

csi_df = pl.DataFrame(report.results_["csi"])
print("Feature stability (CSI):")
print("  PSI < 0.10: green (no shift)")
print("  PSI 0.10-0.25: amber (investigate)")
print("  PSI > 0.25: red (significant shift)")
print()
print(csi_df)

In [ ]:
# Murphy decomposition (when murphy_distribution is set)
# Decomposes model error into: uncertainty + discrimination + miscalibration
# RECALIBRATE when miscalibration dominates; REFIT when discrimination degraded
if report.murphy_available:
    m = report.results_["murphy"]
    print(f"Murphy decomposition ({m['verdict']}):")
    print(f"  Discrimination loss:  {m['discrimination']:.4f}  ({m['discrimination_pct']:.1f}% of total error)")
    print(f"  Miscalibration loss:  {m['miscalibration']:.4f}  ({m['miscalibration_pct']:.1f}% of total error)")
    print(f"  Murphy verdict:       {m['verdict']}")
    print()
    print(f"Combined recommendation: {report.recommendation}")

In [ ]:
# Full results as a flat DataFrame — ready to write to Delta or log to MLflow
print("Full monitoring results:")
report.to_polars()

### Reading the monitoring output

The mild distribution shift we introduced (older drivers, higher mileage) should
produce a green or amber A/E ratio and green Gini — the model is still ranking
correctly, but may need a small intercept adjustment. The CSI on `driver_age`
and `annual_mileage` will flag amber or red, pointing directly at the cause.

In a real quarterly monitoring pack, you would:

1. Write this table to a Delta table alongside the model version and run date
2. Track trends over consecutive quarters
3. Trigger recalibration when A/E exits the green band for two consecutive quarters
4. Trigger a refit project when Gini drops more than 5pp from baseline

## 7. Standalone PSI: quick feature drift check

`MonitoringReport` is the right tool for the full quarterly pack. But sometimes
you just want to check a single feature quickly. The `psi()` function gives you
that in one line.

In [ ]:
from insurance_monitoring.drift import psi, wasserstein_distance

features_to_check = {
    "driver_age": (df_train["driver_age"].values, df_monitor["driver_age"].values),
    "annual_mileage": (df_train["annual_mileage"].values, df_monitor["annual_mileage"].values),
    "vehicle_group": (df_train["vehicle_group"].values, df_monitor["vehicle_group"].values),
    "ncd_years": (df_train["ncd_years"].values, df_monitor["ncd_years"].values),
}

print(f"{'Feature':<20} {'PSI':>8} {'Band':>8} {'Wasserstein':>14}")
print("-" * 55)
for name, (ref, cur) in features_to_check.items():
    p = psi(ref, cur, n_bins=10)
    w = wasserstein_distance(ref, cur)
    band = "green" if p < 0.10 else ("amber" if p < 0.25 else "red")
    print(f"{name:<20} {p:>8.4f} {band:>8} {w:>12.3f}")

print()
print("Wasserstein is in original feature units:")
w_age = wasserstein_distance(df_train["driver_age"].values, df_monitor["driver_age"].values)
print(f"  driver_age has shifted by {w_age:.1f} years on average")

## Summary

In 30 minutes (or about 5 minutes of runtime) we have:

1. **Fitted a CatBoost Poisson frequency model** on 3,500 synthetic UK motor policies

2. **Estimated the causal effect of telematics opt-in** using Double Machine Learning:
   - The naive estimate was biased due to selection (safer drivers opting in)
   - DML recovered the true causal effect (-0.15 log-rate reduction)
   - The difference matters: wrong causal estimates lead to wrong telematics pricing

3. **Added 90% prediction intervals** to the frequency model using split conformal prediction:
   - Distribution-free: no parametric assumptions required
   - Finite-sample coverage guarantee
   - Pearson-weighted score correctly handles the heteroscedasticity of count data

4. **Ran a quarterly monitoring check** against a monitoring period with mild distribution shift:
   - A/E ratio, Gini drift test, and per-feature CSI in one call
   - Murphy decomposition distinguishes RECALIBRATE vs REFIT decisions
   - Output is a flat DataFrame ready to write to Delta or MLflow

---

## Next steps

### Go deeper on each library

| Library | PyPI | GitHub | Full demo notebook |
|---|---|---|---|
| `insurance-causal` | [`pip install insurance-causal`](https://pypi.org/project/insurance-causal/) | [burning-cost/insurance-causal](https://github.com/burning-cost/insurance-causal) | [insurance_causal_demo.py](../notebooks/insurance_causal_demo.py) |
| `insurance-conformal` | [`pip install insurance-conformal`](https://pypi.org/project/insurance-conformal/) | [burning-cost/insurance-conformal](https://github.com/burning-cost/insurance-conformal) | [conformal_prediction_intervals.py](../notebooks/conformal_prediction_intervals.py) |
| `insurance-monitoring` | [`pip install insurance-monitoring`](https://pypi.org/project/insurance-monitoring/) | [burning-cost/insurance-monitoring](https://github.com/burning-cost/insurance-monitoring) | [monitoring_drift_detection.py](../notebooks/monitoring_drift_detection.py) |

### What you didn't see here (but the libraries support)

**insurance-causal:**
- Heterogeneous treatment effects (CATE) by segment using `cate_by_segment()`
- CausalForestDML for nonparametric CATE with BLP/GATES/CLAN inference
- Automatic Debiased ML via Riesz representers for continuous treatments
- FCA-compliant renewal pricing optimisation with price elasticity

**insurance-conformal:**
- Locally-weighted conformal (adaptive-width intervals, ~24% narrower)
- Joint frequency/severity conformal prediction intervals
- Solvency II SCR upper bounds with conformal coverage guarantees
- Conformal risk control for premium sufficiency
- Online conformal with retrospective adjustment (handles distribution shifts)

**insurance-monitoring:**
- TRIPODD feature-interaction-aware drift attribution: which features explain performance degradation
- Anytime-valid A/B testing with mixture SPRT (no pre-specified sample size)
- PIT-based sequential calibration monitoring (anytime-valid type I error control)
- GiniDriftBootstrapTest with governance-ready plot and summary paragraph

### The full ecosystem

The [burning-cost-examples](https://github.com/burning-cost/burning-cost-examples) repository has 40+ Databricks notebooks covering the full pricing workflow: GLM fitting, credibility, fairness audits, survival models, GBM-to-GLM distillation, sequential testing, and more.